# Student finance survey, Utrecht 2024

Reproduces every figure reported in the README.

Fourteen items, self-administered, fielded 18 October 2024. Eleven records, ten usable.

The instrument has three known defects. They are demonstrated here rather than described, in the sections marked **Defect**.

In [1]:
import pandas as pd
from collections import Counter

df = pd.read_csv("responses.csv")
df["start_time"] = pd.to_datetime(df["start_time"])
df["completion_time"] = pd.to_datetime(df["completion_time"])
df["duration_s"] = (df["completion_time"] - df["start_time"]).dt.total_seconds()

print(f"records: {len(df)}")
df[["respondent_id", "age", "duration_s"]]

records: 11


,respondent_id,age,duration_s
0,1,21,190.0
1,2,39,220.0
2,3,34,151.0
3,4,31,600.0
4,5,28,109.0
5,6,31,935.0
6,7,26,385.0
7,8,37,205.0
8,9,24,190.0
9,10,26,130.0


## Cleaning: one record dropped

Respondent 11 opened the form and left after 18 seconds, having answered only the demographic items. Partial records are excluded rather than imputed. Usable base is 10 for every figure below.

In [2]:
df["complete"] = df["taken_finance_course"].notna()
print(df.loc[~df["complete"], ["respondent_id", "duration_s"]].to_string(index=False))

d = df[df["complete"]].copy()
print(f"\nusable n = {len(d)}")
print(f"median completion time: {d['duration_s'].median() / 60:.1f} min")
print(f"age range {d['age'].min()}-{d['age'].max()}, median {d['age'].median()}")

 respondent_id  duration_s
            11        18.0

usable n = 10
median completion time: 3.3 min
age range 21-39, median 29.5


In [3]:
def counts(col):
    """Single-select counts, ordered high to low."""
    return d[col].value_counts()


def multi_counts(col):
    """Multi-select counts. Each option counted once per respondent."""
    c = Counter()
    for v in d[col].dropna():
        for part in str(v).split(";"):
            part = part.strip()
            if part:
                c[part] += 1
    return pd.Series(c).sort_values(ascending=False)

## Sample

Reported as counts throughout. At a base of 10 a percentage implies a precision the data does not have: 55 percent of this sample is five and a half people.

In [4]:
print(counts("student_category"), "\n")
print(counts("financial_understanding"), "\n")
print(counts("taken_finance_course"))

student_category
International Student    5
Dutch Student            2
EU/EEA Student           2
Other (specify)          1
Name: count, dtype: int64 

financial_understanding
Poor    4
Fair    4
Good    2
Name: count, dtype: int64 

taken_finance_course
No     9
Yes    1
Name: count, dtype: int64


## Item 11: why students use or avoid a financial tool

The item the study was written for. Usability outranks both time and indifference.

In [5]:
multi_counts("tool_motivation")

It's hard to use                        6
I don’t have the time                   3
It helps me reach my financial goals    2
I use other methods                     1
I don’t see the need                    1
It helps me stay organized              1
dtype: int64

## Item 13: what would make a tool more engaging

Gamification was offered as an option and selected by nobody. A zero needs no sample size to be worth acting on.

In [6]:
offered = [
    "Gamification (reward system)",
    "Simple and user-friendly design",
    "Personalized financial advice",
    "Integration with banking apps",
    "Visualizations of spending and savings goals",
]
multi_counts("engaging_factors").reindex(offered).fillna(0).astype(int)

Gamification (reward system)                    0
Simple and user-friendly design                 6
Personalized financial advice                   7
Integration with banking apps                   6
Visualizations of spending and savings goals    7
dtype: int64

## Items 7, 8, 10, 12: behaviour, methods, features, stress

In [7]:
print(counts("tracks_expenses"), "\n")
print(multi_counts("methods_used"), "\n")
print(multi_counts("helpful_features"), "\n")
print(counts("stress_frequency"))

tracks_expenses
Sometimes         5
Yes, regularly    3
No, not at all    2
Name: count, dtype: int64 

Banking App                                         6
Spreadsheets (Excel, Google Sheets, Mac Numbers)    3
I don't track my finances.                          2
Financial Apps (Mint, YNAB, etc.)                   1
dtype: int64 

Expense categorization                     8
Budget tracking                            8
Customizable financial reports/Insights    6
Investment tracking                        5
Savings goals                              5
Alerts/reminders for bills and payments    3
Financial tips/education                   2
Debt management features                   2
dtype: int64 

stress_frequency
Often        4
Always       3
Sometimes    2
Rarely       1
Name: count, dtype: int64


## Defect 1: item 3 offered "Other" twice

The income question shipped with two separate options both meaning other, one of them free text. Answers scatter into buckets that cannot be recombined, and one respondent typed a compound answer a single-select item could not hold. Income is not usable as a variable in this dataset.

In [8]:
counts("income_source")

income_source
Parents Support                   3
Loans/Scholarships                2
Other                             2
Other (please specify)            1
Part-Time Job and Student Loan    1
Savings                           1
Name: count, dtype: int64

## Defect 2: item 6 was built as a ranking, so it measures nothing

The intent was to find which financial concepts students find hardest. The item forced every respondent to order all six, so every concept registers for all ten. The output looks like data and carries no signal. A pick-your-top-three, or MaxDiff for a genuine preference ordering, was the right instrument.

In [9]:
flat = multi_counts("challenging_concepts_ranked")
print(flat)
print(f"\ndistinct values across all six options: {flat.nunique()}")
print("every option selected by every respondent, so the item cannot discriminate")

Saving             10
Budgeting          10
Taxes              10
Debt management    10
Loans/Credit       10
Investing          10
dtype: int64

distinct values across all six options: 1
every option selected by every respondent, so the item cannot discriminate


## Defect 3: the 5-point knowledge scale had no anchors

Item 4 asked for a self-rating from very poor to excellent with no point defined. Four said poor and four said fair, but there is no reason to believe two respondents meant the same thing by fair, and no way to check. A behavioural proxy would have been harder to write and worth more.

Below: the same respondents' self-rating against what they actually do. Directional only, and shown to make the point that the two do not have to agree.

In [10]:
pd.crosstab(d["financial_understanding"], d["tracks_expenses"])

tracks_expenses,"No, not at all",Sometimes,"Yes, regularly"
financial_understanding,,,
Fair,1,2,1
Good,0,1,1
Poor,1,2,1


## What this base supports

Ten respondents recruited in one evening from one network, self-selecting. Read the ordering, not the gaps. No difference here would survive a significance test, and none was run.

Two findings are worth carrying forward: usability, not indifference, is the dominant stated barrier, and reward mechanics drew no interest at all.

In [11]:
for _, r in d.iterrows():
    print(f"{r['respondent_id']:>3}  {r['one_change']}")

  1  A prediction model
  2  To gain more control
  3  It could be better if they can teach the students in the app what would make their financial management skills better and improved.
  4  Check my monthly expenses log at first
  5  More investments
  6  record all my expences real-time (like connecting the app to my bank account that can track it real-time)
  7  Be able to manage
  8  Breaking my non-fixed expenses (recreation, food, drinks) and breaking it up into the days left for the month so I know my daily allowance (and it dynamically adjusts based on how much I have or haven’t spent the previous days)
  9  the overview and customization, I have trouble gaining a clear view of everything going on which overwhelms me
 10  I would be more organized with my finances
